# Similarity and Record Linkage

## Objective
Compare Bloom filters and identify matches.

In [4]:
import pandas as pd

df_A = pd.read_pickle("../data/processed/df_A_blocked.pkl")
df_B = pd.read_pickle("../data/processed/df_B_blocked.pkl")

### Dice similarity

In [ ]:
# Calculate similarity score
def dice_similarity(bf1, bf2):
    intersection = (bf1 & bf2).count()
    return 2 * intersection / (bf1.count() + bf2.count() + 1e-10) # 0.0000000001 to prevent division by zero

### Generating pairs

In [ ]:
# def generate_pairs(df_A, df_B, block):
#     pairs = []
#     for val in df_A[block].dropna().unique(): # Get unique values from the blocking column.
#         subA = df_A[df_A[block] == val]
#         subB = df_B[df_B[block] == val]

#         for i in subA.index:
#             for j in subB.index:
#                 pairs.append((i, j))
#     return pairs

# pairs = set()
# for b in ["block1", "block2", "block3"]: # Trying different blocking columns (we dont want to miss potential matches)
#     pairs.update(generate_pairs(df_A, df_B, b))

# matches = []
# for i, j in pairs: # Scoring and filtering matches.
#     sim = dice_similarity(df_A.loc[i,"bloom"], df_B.loc[j,"bloom"])
#     if sim >= 0.85: # If similarity is ≥ 85% consider it a match 
#         matches.append({
#             "id_A": df_A.loc[i,"id"],
#             "id_B": df_B.loc[j,"id"],
#             "sim": sim
#         })

### Generating pairs - updated version

In [1]:
from joblib import Parallel, delayed

def compare_pair(i, j, df_A, df_B):
    bf1 = df_A.loc[i, "bloom"]
    bf2 = df_B.loc[j, "bloom"]

    sim = dice_similarity(bf1, bf2)

    return (i, j, sim)

In [2]:
# Uses all CPUs
def parallel_compare(pairs, df_A, df_B, threshold=0.85):

    results = Parallel(n_jobs=-1)(
        delayed(compare_pair)(i, j, df_A, df_B)
        for i, j in pairs
    )

    matches = [
        (i, j, sim) for i, j, sim in results if sim >= threshold
    ]

    return matches

Candidate pairs → similarity → threshold
<br>Produces predicted matches

In [7]:
matches_df = pd.DataFrame(matches)
matches_df.to_csv("../data/processed/matches.csv", index=False)